<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/5%20-%20Exemplos%20de%20Aplica%C3%A7%C3%A3o/5.2_aprovacao_de_estudantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 5.2 — Aprovação de estudantes: classificação com rede densa

Mesmo pipeline do notebook 5.1, agora num problema com duas features: prever se um estudante passa (1) ou não (0) no exame a partir das horas de estudo (`Study Hours`) e da nota no exame anterior (`Previous Exam Score`). Com só duas features, dá para desenhar os dados e a fronteira de decisão no plano, como fizemos com a Iris nos notebooks 3.1 e 4.2.

## Objetivos

- Aplicar o pipeline do 5.1 (padronização, `Dataset`/`DataLoader`, rede densa com dropout, Adam, `StepLR`, parada antecipada) a um problema de duas features.
- Visualizar os dados e a fronteira de decisão aprendida pela rede.
- Usar o modelo treinado para prever o resultado de um estudante novo, aplicando a mesma padronização do treino.

## Importação das bibliotecas

In [ ]:
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(1111)   # reprodutibilidade: inicialização dos pesos, dropout e embaralhamento
np.random.seed(1111)

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print("device:", device)

## Dados

Usamos o [Student Exam Performance Prediction](https://www.kaggle.com/datasets/mrsimple07/student-exam-performance-prediction), do Kaggle: 500 estudantes. O endereço de download da API do Kaggle funciona sem login (verificado em novembro de 2026); se falhar, baixe o zip pela página e coloque `student_exam_data.csv` em `data/`. O zip traz também `student_exam_data_new.csv`, que é uma cópia idêntica do primeiro arquivo e não é usado.

In [ ]:
# Baixa o zip do Kaggle (não exige login) para ./data e extrai o CSV.
os.makedirs("data", exist_ok=True)
url = "https://www.kaggle.com/api/v1/datasets/download/mrsimple07/student-exam-performance-prediction"
zip_path = "data/student-exam-performance-prediction.zip"
if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("data")

df = pd.read_csv("data/student_exam_data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["Pass/Fail"].value_counts()

As classes são desbalanceadas: cerca de 37% dos estudantes passaram. Vamos ver os dados no plano:

In [ ]:
plt.figure(figsize=(6, 5))
for classe, cor, nome in [(0, "tab:red", "reprovado (0)"), (1, "tab:blue", "aprovado (1)")]:
    sub = df[df["Pass/Fail"] == classe]
    plt.scatter(sub["Study Hours"], sub["Previous Exam Score"], c=cor, s=12, label=nome)
plt.xlabel("horas de estudo")
plt.ylabel("nota no exame anterior")
plt.legend()
plt.show()

In [ ]:
X = df[["Study Hours", "Previous Exam Score"]].values
Y = df["Pass/Fail"].values
print("X:", X.shape, "| Y:", Y.shape)

Separação treino/teste (70%/30%) e padronização ajustada só no treino, como no 5.1. As duas features têm escalas bem diferentes (horas de 1 a 10, notas de 40 a 100); sem padronização, a nota dominaria os gradientes da primeira camada. O conjunto de teste também escolhe a melhor época, então faz o papel de validação.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

X_scaler = StandardScaler()
X_train = X_scaler.fit_transform(X_train)
X_test = X_scaler.transform(X_test)
print("treino:", X_train.shape, "| teste:", X_test.shape)

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
train_dataset = TabularDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset = TabularDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

In [ ]:
train_kwargs = {"batch_size": 100, "shuffle": True}    # embaralha o treino a cada época
test_kwargs = {"batch_size": 100, "shuffle": False}    # a ordem não importa na avaliação
if use_cuda:
    cuda_kwargs = {"num_workers": 1, "pin_memory": True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

train_loader = DataLoader(train_dataset, **train_kwargs)
test_loader = DataLoader(test_dataset, **test_kwargs)
print("mini-batches por época:", len(train_loader))

## Modelo

A mesma rede do 5.1: duas camadas escondidas (128 e 256 neurônios, ReLU), dropout de 0,3 e 2 logits na saída. A entrada agora tem 2 features.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(2, 128)
        self.drop1 = nn.Dropout(0.3)
        self.hidden = nn.Linear(128, 256)
        self.drop2 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, x):
        x = self.input(x)
        x = F.relu(x)
        x = self.drop1(x)
        x = self.hidden(x)
        x = F.relu(x)
        x = self.drop2(x)
        x = self.fc2(x)      # logits
        return x

model = Net()
print(model)
print("parâmetros:", sum(p.numel() for p in model.parameters()))

## Treinamento

Funções `train` e `test` e laço com `StepLR`, parada antecipada e checkpoint, explicados no notebook 5.1.

In [ ]:
def train(model, device, train_loader, optimizer, criterion):
    model.train()                       # liga o dropout
    running_loss = 0.0
    correct = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)            # logits, shape (N, n_classes)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)    # loss é a média do mini-batch; multiplicamos pelo tamanho para somar
        pred = output.argmax(dim=1)                   # classe com o maior logit
        correct += (pred == target).sum().item()
    train_loss = running_loss / len(train_loader.dataset)
    train_acc = 100 * correct / len(train_loader.dataset)
    return train_loss, train_acc

In [ ]:
def test(model, device, test_loader, criterion):
    model.eval()                        # desliga o dropout
    test_loss = 0.0
    correct = 0
    with torch.no_grad():               # avaliação: sem gradientes
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item() * data.size(0)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
    test_loss /= len(test_loader.dataset)
    test_acc = 100 * correct / len(test_loader.dataset)
    return test_loss, test_acc

In [ ]:
n_epochs = 100
lr = 0.001
lr_decay = 10     # a cada lr_decay épocas, o StepLR multiplica a taxa de aprendizado por gamma=0.5
patience = 10      # parada antecipada: para se a acurácia de teste não melhorar por `patience` épocas
patience_counter = 0
best_acc = 0
ckpt_path = "melhor_modelo_estudantes.pt"

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

torch.manual_seed(1111)
model = Net().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()   # recebe logits (aplica log-softmax internamente)
scheduler = StepLR(optimizer, step_size=lr_decay, gamma=0.5)

print("Iniciando treinamento ...")
for epoch in range(1, n_epochs + 1):
    train_loss, train_acc = train(model, device, train_loader, optimizer, criterion)
    test_loss, test_acc = test(model, device, test_loader, criterion)
    scheduler.step()

    print(f"Época {epoch:3d} | treino: custo {train_loss:.4f}, acc {train_acc:.2f}% | teste: custo {test_loss:.4f}, acc {test_acc:.2f}%")
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), ckpt_path)   # state_dict: dicionário com os tensores de pesos e bias
        print("    melhor época até agora: pesos salvos")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter == patience:
            print(f"Parada antecipada na época {epoch}")
            break

In [ ]:
# Recarrega os pesos da melhor época (os da última época podem ser piores)
model.load_state_dict(torch.load(ckpt_path, map_location=device))
test_loss, test_acc = test(model, device, test_loader, criterion)
print(f"Melhor modelo | teste: custo {test_loss:.4f}, acurácia {test_acc:.2f}%")

## Avaliação

In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], label="treino")
plt.plot(range(1, len(history["test_loss"]) + 1), history["test_loss"], label="teste")
plt.title("Função de custo (cross-entropy)")
plt.xlabel("época")
plt.ylabel("custo")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(history["train_acc"]) + 1), history["train_acc"], label="treino")
plt.plot(range(1, len(history["test_acc"]) + 1), history["test_acc"], label="teste")
plt.title("Acurácia")
plt.xlabel("época")
plt.ylabel("acurácia (%)")
plt.legend()
plt.grid(True)

plt.show()

## Exercícios

**1.** Desenhe a fronteira de decisão da rede sobre os pontos de teste. Monte uma grade de pontos cobrindo o plano (horas de estudo × nota anterior), aplique a **mesma** padronização do treino (`X_scaler.transform`), calcule a classe prevista em cada ponto e pinte as regiões com `plt.contourf`. A fronteira é uma reta?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

**2.** Use o modelo para prever o resultado de dois estudantes novos: um que estudou 5 horas e tirou 80 no exame anterior, e outro que estudou 9 horas e tirou 50. Mostre a probabilidade de aprovação de cada um (aplique `F.softmax` aos logits). Lembre-se de padronizar a entrada com o `X_scaler` do treino.

In [ ]:
novos = np.array([[5.0, 80.0],
                  [9.0, 50.0]])
# TODO: escreva sua solução aqui
for (h, n), p in zip(novos, prob_aprovado):
    print(f"{h:.0f} h de estudo, nota anterior {n:.0f}: P(aprovado) = {p:.2f}")

## Aprendizados

- O mesmo pipeline do 5.1 resolve outro problema mudando só a leitura dos dados e o tamanho da entrada.
- Features com escalas diferentes precisam ser padronizadas; dados novos passam pela mesma transformação ajustada no treino.
- Com duas features, a fronteira de decisão mostra o que as camadas escondidas acrescentam: uma fronteira não linear, que uma regressão logística não produz.
- A rede devolve logits; `F.softmax` os converte em probabilidades quando queremos interpretar a previsão.